# KZ Expert Pretraining (Model 2)

Weak supervision pipeline for the KZ (Kol-Zchut) corpus expert using external QA data.

## Step 1: Install Dependencies

In [ ]:
!pip install faiss-cpu

**Data Source**: Kol-Zchut QA dataset from [NNLP-IL/Webiks-Hebrew-RAGbot-KolZchut-QA-Training-DataSet](https://github.com/NNLP-IL/Webiks-Hebrew-RAGbot-KolZchut-QA-Training-DataSet)

Contains Hebrew question-answer pairs for the Kol-Zchut legal/social services domain.

## Step 2: Select Promising Questions

Scores questions by retriever disagreement to find hard cases worth labeling:
- Retrieves candidates via **E5** (dense) and **TF-IDF** (sparse)
- Reranks with **BGE-reranker-v2-m3** (cross-encoder)
- Computes Jaccard disagreement between rankings
- Selects top-N questions with highest disagreement (most ambiguous)

These "hard" questions are most valuable for training.

In [6]:
# === Select the most promising questions using E5, TF-IDF, and BGE-reranker ===
# Outputs:
#   /content/pick_promising/
#     - promising_questions_topN.jsonl  (default N=500)
#     - promising_questions_topN.csv
#     - all_questions_scored.csv        (all questions with metrics)
#
# Tune via env vars at the config block below.

import os, re, json, math, gc, time, hashlib, unicodedata
from pathlib import Path
from typing import List, Dict, Tuple
from collections import defaultdict

import numpy as np
import pandas as pd

# ----------------------- Config -----------------------
KZ_CSV_PATH       = os.getenv("KZ_CSV_PATH", "/content/kz_data.csv")
CORPUS_JSONL      = os.getenv("CORPUS_JSONL", "/content/hsrc_corpus.jsonl")

E5_MODEL_NAME     = os.getenv("E5_MODEL_NAME", "intfloat/multilingual-e5-large")   # bi-encoder
BGE_RERANKER_NAME = os.getenv("BGE_RERANKER_NAME", "BAAI/bge-reranker-v2-m3")      # cross-encoder

EMB_CACHE_DIR     = os.getenv("EMB_CACHE_DIR", "/content/e5_cache")
OUT_DIR           = Path(os.getenv("PROMISE_OUT_DIR", "/content/pick_promising"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
Path(EMB_CACHE_DIR).mkdir(parents=True, exist_ok=True)

# Retrieval sizes
K0_RETRIEVE  = int(os.getenv("K0_RETRIEVE", "30"))   # initial candidates per retriever (E5 & TF-IDF)
K_SCORE      = int(os.getenv("K_SCORE", "30"))        # list size after CE rerank (and for Jaccard sets)
N_SELECT     = int(os.getenv("N_SELECT", "500"))      # how many questions to keep

# Optional limit to test quickly (process only first Q_LIMIT questions)
Q_LIMIT      = int(os.getenv("Q_LIMIT", "0"))         # 0 = no limit

# TF-IDF config
TFIDF_MAX_FEATURES = int(os.getenv("TFIDF_MAX_FEATURES", "200000"))
TFIDF_MAX_DF       = float(os.getenv("TFIDF_MAX_DF", "0.8"))         # keep terms in ≤80% of docs
TFIDF_MIN_DF_RAW   = os.getenv("TFIDF_MIN_DF", "2")                  # "2" or "0.001" etc.

def _parse_min_df(x: str):
    """If x >= 1 -> int(x); if 0 < x <= 1 -> float(x)."""
    try:
        v = float(x)
    except ValueError:
        return 2
    return int(v) if v >= 1 else float(v)

TFIDF_MIN_DF = _parse_min_df(TFIDF_MIN_DF_RAW)

# Weights for the priority score (sum to ~1)
W_E5_CE_DISAGREE     = float(os.getenv("W_E5_CE_DISAGREE", "0.45"))
W_E5_TFIDF_DISAGREE  = float(os.getenv("W_E5_TFIDF_DISAGREE", "0.25"))
W_CE_TFIDF_DISAGREE  = float(os.getenv("W_CE_TFIDF_DISAGREE", "0.10"))
W_CE_AMBIGUITY       = float(os.getenv("W_CE_AMBIGUITY", "0.15"))
W_UNION_SIZE         = float(os.getenv("W_UNION_SIZE", "0.05"))

# Reranker batch
CE_BATCH = int(os.getenv("CE_BATCH", "128"))

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------- Helpers -----------------------
def normalize_he(s: str) -> str:
    if s is None: return ""
    if not isinstance(s, str): s = str(s)
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"[\u0591-\u05C7]", "", s)       # strip niqqud
    s = re.sub(r"\(\/he\/[^)]+\)", " ", s)      # drop Kol-Zchut anchors
    s = re.sub(r"http\S+", " ", s)              # drop URLs
    s = s.replace("\u201c", '"').replace("\u201d", '"').replace("''", '"').replace('""', '"')
    s = re.sub(r"\s+", " ", s).strip()
    return s

def read_kz_questions(csv_path: str) -> List[str]:
    df = pd.read_csv(csv_path, engine="python", sep=None, dtype=str, keep_default_na=False)
    lower_map = {c.lower(): c for c in df.columns}
    q_col = lower_map.get("question")
    if q_col is None:
        raise ValueError(f"CSV must contain 'question' column. Found: {list(df.columns)}")
    qs = df[q_col].astype(str).map(normalize_he).tolist()
    seen, uniq = set(), []
    for q in qs:
        if q and q not in seen:
            seen.add(q); uniq.append(q)
    return uniq

def load_corpus(path: str) -> Dict[str, str]:
    corpus = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            o = json.loads(line)
            uid = o.get("uuid") or o.get("id")
            if uid:
                corpus[uid] = o.get("passage") or o.get("text") or ""
    return corpus

# ----------------------- E5 embeddings + FAISS -----------------------
from transformers import AutoTokenizer, AutoModel
try:
    import faiss
    FAISS = True
except Exception:
    FAISS = False

def _file_sig(p: Path):
    try:
        st = p.stat(); return f"{p.name}|{st.st_size}|{int(st.st_mtime)}"
    except Exception:
        return f"{p.name}|0|0"

def _emb_cache_key(corpus_path: str, model_name: str) -> str:
    base = f"{_file_sig(Path(corpus_path))}|{model_name}|L512"
    return hashlib.sha1(base.encode("utf-8")).hexdigest()[:16]

def _emb_paths(key: str):
    base = f"e5_{key}"
    return (os.path.join(EMB_CACHE_DIR, base + "_embeddings.npy"),
            os.path.join(EMB_CACHE_DIR, base + "_ids.json"),
            os.path.join(EMB_CACHE_DIR, base + "_meta.json"),
            os.path.join(EMB_CACHE_DIR, base + "_index.faiss"))

def load_e5_cache(corpus_path: str, model_name: str):
    e,i,m,fx = _emb_paths(_emb_cache_key(corpus_path, model_name))
    if not (os.path.exists(e) and os.path.exists(i) and os.path.exists(m)):
        return None
    try:
        embs = np.load(e, mmap_mode="r")
        ids  = json.load(open(i, "r", encoding="utf-8"))
        meta = json.load(open(m, "r", encoding="utf-8"))
        return {"emb": embs, "ids": ids, "meta": meta, "faiss": fx}
    except Exception:
        return None

def save_e5_cache(corpus_path: str, model_name: str, ids: List[str], embs: np.ndarray):
    e,i,m,fx = _emb_paths(_emb_cache_key(corpus_path, model_name))
    np.save(e, np.asarray(embs, dtype=np.float32, order="C"))
    json.dump(list(ids), open(i, "w", encoding="utf-8"), ensure_ascii=False)
    json.dump({"model_name": model_name, "num_documents": len(ids), "dim": int(embs.shape[1]),
               "corpus_path": corpus_path, "max_len": 512}, open(m, "w", encoding="utf-8"))
    return fx

class E5:
    def __init__(self, name: str, device: str = DEVICE):
        self.tok = AutoTokenizer.from_pretrained(name, use_fast=True)
        try:
            self.model = AutoModel.from_pretrained(
                name, torch_dtype=(torch.float16 if device=='cuda' else None),
                attn_implementation="sdpa"
            ).to(device)
        except TypeError:
            self.model = AutoModel.from_pretrained(
                name, torch_dtype=(torch.float16 if device=='cuda' else None)
            ).to(device)
        self.model.eval()
        self.device = device

    @staticmethod
    def _prefix(is_query): return "query: " if is_query else "passage: "

    @torch.no_grad()
    def embed(self, texts: List[str], is_query=False, bs=256) -> np.ndarray:
        pref = self._prefix(is_query)
        out=[]
        for i in range(0, len(texts), bs):
            enc = self.tok([pref+t for t in texts[i:i+bs]], padding=True, truncation=True, max_length=512, return_tensors="pt").to(self.device)
            h = self.model(**enc).last_hidden_state
            m = enc['attention_mask'].unsqueeze(-1)
            emb = (h*m).sum(1) / m.sum(1).clamp(min=1)
            emb = torch.nn.functional.normalize(emb, p=2, dim=1)
            out.append(emb.cpu())
        return torch.cat(out, 0).numpy()

def build_faiss_ip(xb: np.ndarray, cache_path: str = None):
    xb = np.asarray(xb, dtype=np.float32, order="C")
    idx = faiss.IndexFlatIP(xb.shape[1])
    idx.add(xb)
    if cache_path:
        faiss.write_index(idx, cache_path)
    return idx

# ----------------------- TF-IDF -----------------------
from sklearn.feature_extraction.text import TfidfVectorizer

def build_tfidf_matrix(texts: List[str]):
    vect = TfidfVectorizer(
        analyzer="word",
        token_pattern=r"(?u)\b\w+\b",
        ngram_range=(1,2),
        min_df=TFIDF_MIN_DF,           # int >=1 or float in (0,1]
        max_df=TFIDF_MAX_DF,           # float in (0,1]
        max_features=TFIDF_MAX_FEATURES,
        sublinear_tf=True,
        dtype=np.float32,
    )
    X = vect.fit_transform(texts)
    return vect, X   # X: (n_docs, n_features) CSR float32

def tfidf_topk(vect: TfidfVectorizer, X_csr, query: str, k: int):
    qv = vect.transform([query])         # (1, F)
    scores = X_csr @ qv.T               # (N, 1) sparse
    scores = np.asarray(scores.todense()).ravel()  # dense 1D
    # top-k indices
    k = min(k, scores.size)
    if k == scores.size:
        order = np.argsort(-scores)
        return order.tolist(), scores
    part = np.argpartition(-scores, kth=k-1)[:k]
    order = part[np.argsort(-scores[part])]
    return order.tolist(), scores

# ----------------------- BGE-reranker (cross-encoder) -----------------------
from transformers import AutoModelForSequenceClassification

class BGEReRanker:
    def __init__(self, name: str, device: str = DEVICE, max_len: int = 512):
        self.device = device
        self.tok = AutoTokenizer.from_pretrained(name, use_fast=True)
        try:
            self.model = AutoModelForSequenceClassification.from_pretrained(
                name, trust_remote_code=True, attn_implementation="sdpa"
            ).to(device)
        except TypeError:
            self.model = AutoModelForSequenceClassification.from_pretrained(
                name, trust_remote_code=True
            ).to(device)
        self.model.eval()
        self.max_len = max_len

    @torch.no_grad()
    def score(self, query: str, passages: List[str], bs: int = CE_BATCH) -> np.ndarray:
        scores=[]
        for i in range(0, len(passages), bs):
            enc = self.tok([query]*min(bs, len(passages)-i),
                           passages[i:i+bs],
                           padding=True, truncation=True, max_length=self.max_len, return_tensors="pt").to(self.device)
            with torch.autocast(device_type=("cuda" if self.device=="cuda" else "cpu"),
                                dtype=torch.float16 if self.device=="cuda" else torch.bfloat16,
                                enabled=(self.device=="cuda")):
                logits = self.model(**enc).logits
            if logits.dim()==2 and logits.shape[1]==1:
                s = logits.squeeze(-1).float()
            elif logits.dim()==2 and logits.shape[1]==2:
                s = logits[:,1].float()
            else:
                s = logits.view(-1).float()
            scores.append(s.cpu().numpy())
        return np.concatenate(scores, axis=0)

# ----------------------- Scoring utilities -----------------------
def jaccard(a_ids: List[str], b_ids: List[str]) -> float:
    sa, sb = set(a_ids), set(b_ids)
    if not sa and not sb: return 1.0
    return len(sa & sb) / max(1, len(sa | sb))

# ----------------------- Main selection -----------------------
def select_promising_questions():
    print("[LOAD] corpus …")
    corpus = load_corpus(CORPUS_JSONL)
    doc_ids = list(corpus.keys())
    doc_texts = [corpus[d] for d in doc_ids]
    print(f"[DATA] corpus docs={len(doc_ids):,}")

    print("[LOAD] KZ unique questions …")
    questions = read_kz_questions(KZ_CSV_PATH)
    if Q_LIMIT > 0:
        questions = questions[:Q_LIMIT]
    print(f"[DATA] unique questions={len(questions):,}")

    # ---- TF-IDF matrix
    print("[TFIDF] building matrix …")
    vect, X_tfidf = build_tfidf_matrix(doc_texts)

    # ---- E5 embeddings + FAISS
    print("[E5] preparing embeddings/index …")
    cache = load_e5_cache(CORPUS_JSONL, E5_MODEL_NAME)
    e5 = E5(E5_MODEL_NAME, device=DEVICE)

    use_faiss = FAISS
    index = None
    Xdot = None

    if cache is None or cache["ids"] != doc_ids:
        print("[E5] embedding corpus (one-time) …")
        X_e5 = e5.embed(doc_texts, is_query=False, bs=128)
        faiss_path = save_e5_cache(CORPUS_JSONL, E5_MODEL_NAME, doc_ids, X_e5)
        if use_faiss:
            index = build_faiss_ip(X_e5, cache_path=faiss_path)
        else:
            Xdot = X_e5.T
        del X_e5; gc.collect()
    else:
        X_e5_mem = cache["emb"]   # memmap
        if use_faiss and os.path.exists(cache["faiss"]):
            index = faiss.read_index(cache["faiss"])
        elif use_faiss:
            index = build_faiss_ip(np.asarray(X_e5_mem), cache_path=cache["faiss"])
        else:
            Xdot = np.asarray(X_e5_mem).T

    # ---- BGE reranker
    print("[BGE] loading reranker …")
    ce = BGEReRanker(BGE_RERANKER_NAME, device=DEVICE, max_len=512)

    rows=[]
    t0 = time.perf_counter()

    for qi, q in enumerate(questions, 1):
        # E5 retrieve
        qv = e5.embed([q], is_query=True, bs=1).astype(np.float32)  # (1, D)
        if use_faiss:
            D_e5, I_e5 = index.search(qv, min(K0_RETRIEVE, len(doc_ids)))
            e5_idx = I_e5[0].tolist()
        else:
            sims = (qv @ Xdot)[0]
            e5_idx = np.argsort(sims)[::-1][:K0_RETRIEVE].tolist()

        # TF-IDF retrieve
        tf_idx, _tf_scores = tfidf_topk(vect, X_tfidf, q, K0_RETRIEVE)

        # Pool union
        pool_idx = list(dict.fromkeys(e5_idx + tf_idx))  # preserve order, dedup
        pool_texts = [corpus[doc_ids[i]] for i in pool_idx]

        # CE rerank
        ce_scores = ce.score(q, pool_texts, bs=CE_BATCH)
        order_ce  = np.argsort(-ce_scores)
        ce_top    = [pool_idx[i] for i in order_ce[:K_SCORE]]
        e5_top    = e5_idx[:K_SCORE]
        tf_top    = tf_idx[:K_SCORE]

        # Metrics
        e5_top_ids = [doc_ids[i] for i in e5_top]
        tf_top_ids = [doc_ids[i] for i in tf_top]
        ce_top_ids = [doc_ids[i] for i in ce_top]

        j_e5_ce     = jaccard(e5_top_ids, ce_top_ids)
        j_e5_tfidf  = jaccard(e5_top_ids, tf_top_ids)
        j_ce_tfidf  = jaccard(ce_top_ids, tf_top_ids)

        disagree_e5_ce    = 1.0 - j_e5_ce
        disagree_e5_tfidf = 1.0 - j_e5_tfidf
        disagree_ce_tfidf = 1.0 - j_ce_tfidf

        # CE ambiguity: 1 - normalized top gap
        if len(order_ce) >= 2:
            s1, s2 = ce_scores[order_ce[0]], ce_scores[order_ce[1]]
            gap = float(s1 - s2)
            denom = float(max(abs(s1), 1e-6))
            ce_ambig = float(1.0 - max(0.0, min(1.0, gap/denom)))
        else:
            ce_ambig = 0.5

        # Union size normalized (encourages diversity)
        union_ids = set(e5_top_ids) | set(tf_top_ids) | set(ce_top_ids)
        union_norm = len(union_ids) / float(3 * K_SCORE)

        # Priority score
        score = (
            W_E5_CE_DISAGREE    * disagree_e5_ce +
            W_E5_TFIDF_DISAGREE * disagree_e5_tfidf +
            W_CE_TFIDF_DISAGREE * disagree_ce_tfidf +
            W_CE_AMBIGUITY      * ce_ambig +
            W_UNION_SIZE        * union_norm
        )

        rows.append({
            "question": q,
            "score": float(score),
            "metrics": {
                "j_e5_ce": float(j_e5_ce),
                "j_e5_tfidf": float(j_e5_tfidf),
                "j_ce_tfidf": float(j_ce_tfidf),
                "ce_ambiguity": float(ce_ambig),
                "union_norm": float(union_norm),
            },
            "e5_top_docids": e5_top_ids,
            "tf_top_docids": tf_top_ids,
            "ce_top_docids": ce_top_ids,
        })

        if qi % 50 == 0:
            dt = time.perf_counter() - t0
            print(f"[{qi}/{len(questions)}] avg {qi/dt:.2f} q/s")

    # Rank questions by priority
    dfq = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
    top_df = dfq.head(N_SELECT).copy()

    # Save CSVs
    dfq_small = dfq.drop(columns=["e5_top_docids","tf_top_docids","ce_top_docids"])
    dfq_small.to_csv(OUT_DIR / "all_questions_scored.csv", index=False, encoding="utf-8")
    top_df.drop(columns=["e5_top_docids","tf_top_docids","ce_top_docids"]).to_csv(
        OUT_DIR / f"promising_questions_top{N_SELECT}.csv", index=False, encoding="utf-8"
    )

    # Save JSONL for the top N
    out_jsonl = OUT_DIR / f"promising_questions_top{N_SELECT}.jsonl"
    with open(out_jsonl, "w", encoding="utf-8") as f:
        for _, r in top_df.iterrows():
            f.write(json.dumps({
                "question": r["question"],
                "priority_score": float(r["score"]),
                "metrics": r["metrics"],
                "e5_top_docids": r["e5_top_docids"],
                "tf_top_docids": r["tf_top_docids"],
                "ce_top_docids": r["ce_top_docids"],
            }, ensure_ascii=False) + "\n")

    print(f"[DONE] Wrote ranked lists to: {OUT_DIR}")
    print(f"Top 5 (question | score):")
    for i in range(min(5, len(top_df))):
        print(f"{i+1:>2}. {top_df.loc[i,'question'][:140]} | {top_df.loc[i,'score']:.4f}")

# ------------- Run -------------
if __name__ == "__main__":
    select_promising_questions()


[LOAD] corpus …
[DATA] corpus docs=127,731
[LOAD] KZ unique questions …
[DATA] unique questions=2,950
[TFIDF] building matrix …
[E5] preparing embeddings/index …
[BGE] loading reranker …
[50/2950] avg 2.99 q/s
[100/2950] avg 3.00 q/s
[150/2950] avg 3.01 q/s
[200/2950] avg 3.00 q/s
[250/2950] avg 3.00 q/s
[300/2950] avg 2.99 q/s
[350/2950] avg 2.99 q/s
[400/2950] avg 2.98 q/s
[450/2950] avg 2.97 q/s
[500/2950] avg 2.97 q/s
[550/2950] avg 2.96 q/s
[600/2950] avg 2.96 q/s
[650/2950] avg 2.96 q/s
[700/2950] avg 2.96 q/s
[750/2950] avg 2.96 q/s
[800/2950] avg 2.96 q/s
[850/2950] avg 2.96 q/s
[900/2950] avg 2.96 q/s
[950/2950] avg 2.96 q/s
[1000/2950] avg 2.96 q/s
[1050/2950] avg 2.96 q/s
[1100/2950] avg 2.96 q/s
[1150/2950] avg 2.96 q/s
[1200/2950] avg 2.96 q/s
[1250/2950] avg 2.96 q/s
[1300/2950] avg 2.96 q/s
[1350/2950] avg 2.96 q/s
[1400/2950] avg 2.96 q/s
[1450/2950] avg 2.96 q/s
[1500/2950] avg 2.96 q/s
[1550/2950] avg 2.96 q/s
[1600/2950] avg 2.96 q/s
[1650/2950] avg 2.96 q/s
[1700/29

## Step 3: Build KZ Pretrain Groups

Combines CSV answers (positives) with RRF-fused candidates (unlabeled negatives):
- CSV answers labeled as 3 (strong positive)
- Candidates from E5/TF-IDF/CE ranked via RRF, labeled as 0

Outputs train/val JSONL splits for Stage 2 training.

In [ ]:
# === Build KZ pretrain groups from: CSV (positives) + candidate doc_ids JSONL (unlabeled negatives) ===
# Inputs:
#   1) CSV with columns: question, paragraph, (optional) link, doc_id  -> answers=positives
#   2) Candidates JSONL, one JSON per line:
#        {
#          "question": "...",
#          "priority_score": ...,
#          "e5_top_docids": [...],
#          "tf_top_docids": [...],
#          "ce_top_docids": [...]
#        }
#   3) HSRC corpus JSONL: { "uuid": <doc_id>, "passage": <text> }
#
# Outputs (Stage-2 compatible):
#   stage1/groups_pretrain_kz_train.jsonl
#   stage1/groups_pretrain_kz_val.jsonl
#   stage1/pretrain_kz_train_query_uuids.txt
#   stage1/pretrain_kz_val_query_uuids.txt
#   stage1/pretrain_kz_query_uuid_splits.json

import os, json, re, math, random, hashlib, gc
from pathlib import Path
from typing import List, Dict, Tuple, Set
import numpy as np
import pandas as pd

# ----------------------- Config -----------------------
KZ_CSV_PATH       = os.getenv("KZ_CSV_PATH", "/content/kz_unlabeled.csv")
CANDIDATES_JSONL  = os.getenv("CANDIDATES_JSONL", "/content/kz_candidates.jsonl")  # your JSONL with e5/tf/ce lists
CORPUS_JSONL      = os.getenv("CORPUS_JSONL", "/content/hsrc_corpus.jsonl")

OUTPUT_DIR        = os.getenv("OUTPUT_DIR", "/content/ce_ft/bge-reranker-v2-m3_t50_fp16")
STAGE1_DIR        = os.getenv("STAGE1_DIR", os.path.join(OUTPUT_DIR, "stage1"))

# Group composition
K_PRE             = int(os.getenv("K_PRE", "32"))    # total items per group (pos + neg)
MAX_POS_PER_Q     = int(os.getenv("MAX_POS_PER_Q", "4"))
NEG_MAX_PER_Q     = int(os.getenv("NEG_MAX_PER_Q", str(max(1, K_PRE - MAX_POS_PER_Q))))
RRF_K             = int(os.getenv("RRF_K", "60"))    # RRF stabilizer
SEED              = int(os.getenv("SEED", "42"))
VAL_SIZE          = float(os.getenv("VAL_SIZE", "0.20"))  # split by question
CASE_NAME         = "mafat_retrieval_kz_corpus"

Path(STAGE1_DIR).mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)

def _norm_text(s: str) -> str:
    if s is None: return ""
    s = str(s).replace("\u200f"," ").replace("\u200e"," ")
    s = re.sub(r"[“”„״]", '"', s)
    s = re.sub(r"[‘’׳]", "'", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _hash_uuid(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8")).hexdigest()

def _write_jsonl(path: str, items: List[Dict]):
    with open(path, "w", encoding="utf-8") as f:
        for o in items:
            f.write(json.dumps(o, ensure_ascii=False) + "\n")

def _write_lines(path: str, lines: List[str]):
    with open(path, "w", encoding="utf-8") as f:
        for ln in lines:
            f.write(str(ln) + "\n")

print(f"[SETUP] CSV={KZ_CSV_PATH}")
print(f"[SETUP] CANDIDATES_JSONL={CANDIDATES_JSONL}")
print(f"[SETUP] CORPUS_JSONL={CORPUS_JSONL}")
print(f"[SETUP] OUT={STAGE1_DIR}  K_PRE={K_PRE}  MAX_POS_PER_Q={MAX_POS_PER_Q}  VAL_SIZE={VAL_SIZE}")

# ----------------------- Load corpus -----------------------
uuid2text: Dict[str, str] = {}
with open(CORPUS_JSONL, "r", encoding="utf-8") as f:
    for ln in f:
        if not ln.strip(): continue
        o = json.loads(ln)
        uid = o.get("uuid") or o.get("id")
        if not uid: continue
        txt = o.get("passage") or o.get("text") or ""
        uuid2text[uid] = txt
print(f"[LOAD] Corpus docs: {len(uuid2text):,}")

# ----------------------- Load candidates JSONL -----------------------
# Build: q_norm -> ordered union of candidate doc_ids via RRF over (e5, tf, ce)
def _rrf_rank(lists: List[List[str]], k: int, rrf_k: int = 60) -> List[str]:
    rank_maps = []
    for L in lists:
        if not L:
            rank_maps.append({})
            continue
        rank_maps.append({d: r for r, d in enumerate(L)})
    all_ids: Set[str] = set()
    for rm in rank_maps:
        all_ids.update(rm.keys())
    scored = []
    for d in all_ids:
        s = 0.0
        for rm in rank_maps:
            r = rm.get(d)
            if r is not None:
                s += 1.0 / (rrf_k + r)
        scored.append((d, s))
    scored.sort(key=lambda x: x[1], reverse=True)
    return [d for d, _ in scored[:k]]

cand_by_q: Dict[str, List[str]] = {}
kept, missing = 0, 0
with open(CANDIDATES_JSONL, "r", encoding="utf-8") as f:
    for ln in f:
        if not ln.strip(): continue
        o = json.loads(ln)
        q = _norm_text(o.get("question", ""))
        if not q:
            continue
        e5 = o.get("e5_top_docids", []) or []
        tf = o.get("tf_top_docids", []) or []
        ce = o.get("ce_top_docids", []) or []
        # RRF union; keep only doc_ids present in corpus
        ordered = [d for d in _rrf_rank([e5, tf, ce], k=max(K_PRE*3, 200), rrf_k=RRF_K) if d in uuid2text]
        if not ordered:
            missing += 1
            continue
        cand_by_q[q] = ordered
        kept += 1
print(f"[LOAD] Candidates: {kept} questions with candidates, {missing} without (skipped)")

# ----------------------- Load CSV → answers per question -----------------------
# Robust read (multiline fields)
try:
    df = pd.read_csv(
        KZ_CSV_PATH,
        engine="python",
        sep=None,
        quotechar='"',
        doublequote=True,
        escapechar="\\",
        on_bad_lines="skip",
        dtype={"question": str, "paragraph": str, "link": str, "doc_id": str},
        keep_default_na=False,
    )
except Exception:
    df = pd.read_csv(
        KZ_CSV_PATH, engine="python", sep=",", quotechar='"', doublequote=True,
        escapechar="\\", on_bad_lines="skip",
        dtype={"question": str, "paragraph": str, "link": str, "doc_id": str},
        keep_default_na=False,
    )
df.columns = [c.strip().lower() for c in df.columns]
if "question" not in df.columns or "paragraph" not in df.columns:
    raise ValueError(f"CSV must contain 'question' and 'paragraph' columns; got {df.columns.tolist()}")

df["question"]  = df["question"].map(_norm_text)
df["paragraph"] = df["paragraph"].map(_norm_text)
if "doc_id" not in df.columns:
    df["doc_id"] = [f"row_{i}" for i in range(len(df))]
else:
    df["doc_id"] = df["doc_id"].astype(str)

# keep only rows whose normalized question exists in candidates (the curated ~500)
df = df[df["question"].map(lambda q: q in cand_by_q)].copy()
df = df[(df["question"].str.len() > 0) & (df["paragraph"].str.len() > 0)]
df.reset_index(drop=True, inplace=True)

# Gather answers per question
answers_by_q: Dict[str, List[Tuple[str, str]]] = {}  # q -> list of (pid, text)
for i, row in df.iterrows():
    q  = row["question"]
    pid = f"csvans_{row['doc_id']}_{i}"        # synthetic pid for CSV answers
    txt = row["paragraph"]
    answers_by_q.setdefault(q, [])
    # dedupe identical texts
    if txt and all(txt != t for _, t in answers_by_q[q]):
        answers_by_q[q].append((pid, txt))

print(f"[CSV] Questions with answers (intersection): {len(answers_by_q)}")

# ----------------------- Build groups -----------------------
def build_group(q: str) -> Dict:
    # positives: CSV answers (cap)
    pos = answers_by_q.get(q, [])
    if not pos:
        return {}
    if len(pos) > MAX_POS_PER_Q:
        pos = sorted(random.sample(pos, MAX_POS_PER_Q))
    pos_pids  = [p for p, _ in pos]
    pos_texts = [t for _, t in pos]
    pos_labels= [3]*len(pos)  # strong positive for ListNet (gain=7)

    # negatives: RRF-ordered candidates from corpus (unlabeled)
    cand_ids = cand_by_q.get(q, [])
    neg_take = max(0, min(NEG_MAX_PER_Q, K_PRE - len(pos)))
    neg_ids  = []
    for d in cand_ids:
        if d in neg_ids:         # dedupe
            continue
        neg_ids.append(d)
        if len(neg_ids) >= neg_take:
            break
    # map to text; drop any missing (should be filtered already)
    neg_pids  = []
    neg_texts = []
    for d in neg_ids:
        txt = uuid2text.get(d)
        if not txt: continue
        neg_pids.append(d)
        neg_texts.append(txt)
    neg_labels = [0]*len(neg_pids)

    # final lists
    pids   = pos_pids + neg_pids
    texts  = pos_texts + neg_texts
    labels = pos_labels + neg_labels

    return {
        "query": q,
        "query_uuid": _hash_uuid("kz_csv::" + q),
        "case_name": CASE_NAME,
        "pids": pids,
        "texts": texts,
        "labels": labels,
    }

groups: List[Dict] = []
for q in sorted(answers_by_q.keys()):
    if q not in cand_by_q:
        continue
    g = build_group(q)
    if g and len(g["texts"]) >= 2:  # at least 1 pos + 1 neg
        groups.append(g)

print(f"[BUILD] Built {len(groups)} groups (K_PRE target={K_PRE}, MAX_POS_PER_Q={MAX_POS_PER_Q}, NEG_MAX_PER_Q={NEG_MAX_PER_Q})")

# ----------------------- Split (by question) -----------------------
idxs = list(range(len(groups)))
random.shuffle(idxs)
cut = max(1, int(len(groups) * (1.0 - VAL_SIZE)))
groups_train = [groups[i] for i in idxs[:cut]]
groups_val   = [groups[i] for i in idxs[cut:]]

print(f"[SPLIT] train={len(groups_train)}  val={len(groups_val)}  (VAL_SIZE={VAL_SIZE})")

# ----------------------- Save -----------------------
pretrain_train_path = os.path.join(STAGE1_DIR, "groups_pretrain_kz_train.jsonl")
pretrain_val_path   = os.path.join(STAGE1_DIR, "groups_pretrain_kz_val.jsonl")
_write_jsonl(pretrain_train_path, groups_train)
_write_jsonl(pretrain_val_path, groups_val)

train_qids = [g["query_uuid"] for g in groups_train]
val_qids   = [g["query_uuid"] for g in groups_val]

train_qids_txt = os.path.join(STAGE1_DIR, "pretrain_kz_train_query_uuids.txt")
val_qids_txt   = os.path.join(STAGE1_DIR, "pretrain_kz_val_query_uuids.txt")
splits_json    = os.path.join(STAGE1_DIR, "pretrain_kz_query_uuid_splits.json")
_write_lines(train_qids_txt, train_qids)
_write_lines(val_qids_txt, val_qids)
with open(splits_json, "w", encoding="utf-8") as f:
    json.dump({"train_query_uuids": train_qids, "val_query_uuids": val_qids}, f, ensure_ascii=False, indent=2)

print(f"[SAVE] groups_pretrain_kz_train.jsonl → {pretrain_train_path}")
print(f"[SAVE] groups_pretrain_kz_val.jsonl   → {pretrain_val_path}")
print(f"[SAVE] pretrain_kz_train_query_uuids.txt (n={len(train_qids)})")
print(f"[SAVE] pretrain_kz_val_query_uuids.txt   (n={len(val_qids)})")
print(f"[SAVE] pretrain_kz_query_uuid_splits.json")

gc.collect()


## Step 4: Train KZ Expert on Pretrain Groups

Fine-tunes BGE-reranker-v2-m3 on the generated groups:
- **Positives**: CSV answers (label=3, gain=7)
- **Negatives**: RRF-fused candidates (label=0)

Uses ListNet loss with τ=1.1, saves best checkpoint by validation NDCG@20.

In [ ]:
# === Train CE on generated KZ pretrain groups (answers = positives, candidates = unlabeled) ===
import os, json, math, random, time, gc, shutil
from pathlib import Path
from typing import List, Dict, Tuple, Union
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_polynomial_decay_schedule_with_warmup

# ----------------------- Config -----------------------
OUTPUT_DIR    = os.getenv("OUTPUT_DIR", "/content/ce_ft/bge-reranker-v2-m3_t50_fp16")
STAGE1_DIR    = os.getenv("STAGE1_DIR", os.path.join(OUTPUT_DIR, "stage1"))

# Paths to the generated groups
GROUPS_TRAIN  = os.getenv("GROUPS_PRETRAIN_TRAIN", os.path.join(STAGE1_DIR, "groups_pretrain_kz_train.jsonl"))
GROUPS_VAL    = os.getenv("GROUPS_PRETRAIN_VAL",   os.path.join(STAGE1_DIR, "groups_pretrain_kz_val.jsonl"))

# Where to save the model
RUN_DIR       = os.getenv("RUN_DIR", os.path.join(OUTPUT_DIR, "ce_kz_pretrain"))
CHECKPOINTS   = os.path.join(RUN_DIR, "checkpoints")
BEST_DIR      = os.path.join(RUN_DIR, "best")
METRICS_LOG   = os.path.join(RUN_DIR, "metrics.jsonl")

# Base CE to fine-tune (can be a local folder or HF id)
CE_MODEL_NAME = os.getenv("CE_MODEL_NAME", "BAAI/bge-reranker-v2-m3")

# Train/eval hyperparams
EPOCHS          = int(os.getenv("EPOCHS", "2"))
LR              = float(os.getenv("LR", "8e-6"))
BATCH_GROUPS    = int(os.getenv("BATCH_GROUPS", "2"))      # number of groups per step
GRAD_ACCUM      = int(os.getenv("GRAD_ACCUM", "1"))
MAX_LEN         = int(os.getenv("MAX_LEN", "448"))
WEIGHT_DECAY    = float(os.getenv("WEIGHT_DECAY", "0.02"))
CLIP_NORM       = float(os.getenv("CLIP_NORM", "1.0"))
TAU             = float(os.getenv("TAU", "1.1"))
VAL_EVAL_K      = int(os.getenv("VAL_EVAL_K", "20"))
EVAL_BATCH_PAIRS= int(os.getenv("EVAL_BATCH_PAIRS", "192"))
PAD_TO_MULTIPLE = int(os.getenv("PAD_TO_MULTIPLE_OF", "8"))
RESET_HEAD      = int(os.getenv("RESET_HEAD_FOR_SUP", "0"))   # 1 = reinit classification head

SEED            = int(os.getenv("SEED", "42"))

# ----------------------- Setup -----------------------
def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
Path(RUN_DIR).mkdir(parents=True, exist_ok=True)
Path(CHECKPOINTS).mkdir(parents=True, exist_ok=True)

print(f"[CONFIG] device={device}  base={CE_MODEL_NAME}")
print(f"[DATA]   train={GROUPS_TRAIN}")
print(f"[DATA]   val  ={GROUPS_VAL}")
print(f"[OUT]    run_dir={RUN_DIR}")

# ----------------------- IO -----------------------
def _read_jsonl(path: str) -> List[Dict]:
    items=[]
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                items.append(json.loads(line))
    return items

assert Path(GROUPS_TRAIN).exists(), f"Missing groups file: {GROUPS_TRAIN}"
assert Path(GROUPS_VAL).exists(),   f"Missing groups file: {GROUPS_VAL}"

groups_train = _read_jsonl(GROUPS_TRAIN)
groups_val   = _read_jsonl(GROUPS_VAL)
print(f"[LOAD] groups: train={len(groups_train)}  val={len(groups_val)}")

# ----------------------- Dataset / Collate -----------------------
class ListwiseDataset(torch.utils.data.Dataset):
    def __init__(self, groups): self.groups = groups
    def __len__(self): return len(self.groups)
    def __getitem__(self, i):
        g = self.groups[i]
        return g["query"], g["texts"], g["labels"], g["pids"]

def collate_listwise(batch, tokenizer, max_len=512):
    Q, P, gains, spans, PIDS = [], [], [], [], []
    cur = 0
    for q, texts, labs, pids in batch:
        Q += [q]*len(texts)
        P += texts
        # labels are integers (0/3). ListNet uses gains = 2^label - 1
        gains += [float((2**int(l))-1) for l in labs]
        PIDS += pids
        spans.append((cur, cur+len(texts)))
        cur += len(texts)
    enc = tokenizer(Q, P, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
    return enc, torch.tensor(gains, dtype=torch.float32), spans, PIDS

# ----------------------- Loss -----------------------
def listnet_loss(scores: torch.Tensor, gains: torch.Tensor, spans, tau=1.0):
    scores = scores.float(); gains = gains.float()
    loss_terms = []
    for s, e in spans:
        g = gains[s:e]
        if torch.count_nonzero(g) == 0: continue
        p_t = g / (g.sum() + 1e-12)
        log_p_s = torch.log_softmax(scores[s:e] / tau, dim=0)
        loss_terms.append(-(p_t * log_p_s).sum())
    return torch.stack(loss_terms).mean() if loss_terms else scores.sum() * 0.0

# ----------------------- Eval (fast, pretokenized) -----------------------
def _mean_or_zero(vals): return float(np.mean(vals)) if len(vals) > 0 else 0.0

@torch.no_grad()
def evaluate_ndcg_groups(model, pretok: dict, k=20, eval_bs_pairs=128) -> List[float]:
    model.eval()
    enc_full = pretok["enc"]; spans = pretok["spans"]; rels_all = pretok["rels"]
    if not spans or rels_all.numel() == 0: return []
    N = enc_full["input_ids"].size(0)
    scores = torch.empty(N, dtype=torch.float32)
    start = 0
    while start < N:
        end = min(start + eval_bs_pairs, N)
        sl = slice(start, end)
        inputs = {k: v[sl].to(device, non_blocking=True) for k,v in enc_full.items()}
        ctx = (torch.autocast(device_type="cuda", dtype=torch.float16) if device=="cuda"
               else torch.cpu.amp.autocast(enabled=False))
        with torch.inference_mode(), ctx:
            logits = model(**inputs).logits
            if logits.dim()==2 and logits.shape[1]==1: s = logits.squeeze(-1).float()
            elif logits.dim()==2 and logits.shape[1]==2: s = logits[:,1].float()
            else: s = logits.view(-1).float()
        scores[sl] = s.detach().cpu()
        start = end

    denom = 1.0 / np.log2(np.arange(2, VAL_EVAL_K + 2))
    s_np = scores.numpy(); r_np = rels_all.numpy()
    ndcgs = []
    for (st, ed) in spans:
        group_scores = s_np[st:ed]
        group_rels   = r_np[st:ed]
        if group_rels.size == 0:
            ndcgs.append(0.0); continue
        order = np.argsort(-group_scores)
        rel_sorted = group_rels[order][:VAL_EVAL_K]
        dcg = ((np.power(2.0, rel_sorted, dtype=np.float64) - 1.0) * denom[:len(rel_sorted)]).sum()
        ideal = np.sort(group_rels)[::-1][:VAL_EVAL_K]
        idcg = ((np.power(2.0, ideal, dtype=np.float64) - 1.0) * denom[:len(ideal)]).sum()
        ndcgs.append(0.0 if idcg <= 0.0 else float(dcg / idcg))
    return ndcgs

def build_val_pretok(groups, tokenizer, max_len=512, pad_multi=8):
    all_input_ids, all_attn, all_ttids = [], [], []
    all_rels, all_pids, spans = [], [], []
    cur = 0
    for g in groups:
        q, texts, labels, pids = g["query"], g["texts"], g["labels"], g["pids"]
        enc = tokenizer([q]*len(texts), texts,
                        padding="max_length", truncation=True, max_length=max_len,
                        pad_to_multiple_of=pad_multi, return_tensors="pt")
        all_input_ids.append(enc["input_ids"])
        all_attn.append(enc["attention_mask"])
        if "token_type_ids" in enc: all_ttids.append(enc["token_type_ids"])
        all_rels.extend([int(l) for l in labels]); all_pids.extend(pids)
        spans.append((cur, cur + len(texts))); cur += len(texts)
    if all_input_ids:
        input_ids = torch.cat(all_input_ids, dim=0)
        attention_mask = torch.cat(all_attn, dim=0)
        enc_full = {"input_ids": input_ids, "attention_mask": attention_mask}
        if len(all_ttids) > 0: enc_full["token_type_ids"] = torch.cat(all_ttids, dim=0)
        for k in list(enc_full.keys()): enc_full[k] = enc_full[k].pin_memory()
        rels = torch.tensor(all_rels, dtype=torch.int16)
    else:
        L = max_len
        enc_full = {"input_ids": torch.empty(0, L, dtype=torch.long),
                    "attention_mask": torch.empty(0, L, dtype=torch.long)}
        rels = torch.empty(0, dtype=torch.int16)
    return {"enc": enc_full, "spans": spans, "pids": all_pids, "rels": rels}

# ----------------------- Save helpers -----------------------
def _dir_size_bytes(path: str) -> int:
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            fp = os.path.join(root, f)
            try: total += os.path.getsize(fp)
            except OSError: pass
    return total

def _human_bytes(n: int) -> str:
    units = ["B","KB","MB","GB","TB"]
    i = 0; v = float(n)
    while v >= 1024.0 and i < len(units)-1: v /= 1024.0; i += 1
    return f"{v:.2f} {units[i]}"

def _save_fp16_and_report(model, tokenizer, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    orig_dtype = next(model.parameters()).dtype
    try:
        model.to(dtype=torch.float16)
        model.save_pretrained(out_dir, safe_serialization=True)  # safetensors
        tokenizer.save_pretrained(out_dir)
    finally:
        model.to(dtype=orig_dtype)
    sz = _dir_size_bytes(out_dir)
    print(f"[SAVE] → {out_dir}  ({_human_bytes(sz)})")
    return out_dir

def _write_metrics_line(path: str, row: Dict):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

def _replace_dir(dst: str):
    if os.path.isdir(dst): shutil.rmtree(dst)
    os.makedirs(dst, exist_ok=True)

# ----------------------- Build loaders -----------------------
tok = AutoTokenizer.from_pretrained(CE_MODEL_NAME)
try:
    model = AutoModelForSequenceClassification.from_pretrained(
        CE_MODEL_NAME, trust_remote_code=True, attn_implementation="sdpa"
    ).to(device)
except TypeError:
    model = AutoModelForSequenceClassification.from_pretrained(
        CE_MODEL_NAME, trust_remote_code=True
    ).to(device)
model = model.float()

# Optional: reset head to adapt quickly from weak labels → supervised
if RESET_HEAD == 1:
    head = getattr(model, "score", None) or getattr(model, "classifier", None)
    if isinstance(head, torch.nn.Linear):
        torch.nn.init.normal_(head.weight, std=0.02)
        if head.bias is not None:
            torch.nn.init.zeros_(head.bias)
        print("[INIT] Reset CE head.")

train_ds = ListwiseDataset(groups_train)
train_loader = torch.utils.data.DataLoader(
    train_ds,
    batch_size=BATCH_GROUPS,
    shuffle=True,
    collate_fn=lambda b: collate_listwise(b, tok, MAX_LEN),
    pin_memory=torch.cuda.is_available(),
    num_workers=2 if torch.cuda.is_available() else 0,
    persistent_workers=torch.cuda.is_available()
)

pretok_val = build_val_pretok(groups_val, tok, max_len=MAX_LEN, pad_multi=PAD_TO_MULTIPLE)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, eps=1e-8, betas=(0.9, 0.999)
)
updates_per_epoch = max(1, math.ceil(len(train_loader) / max(1, GRAD_ACCUM)))
num_update_steps  = max(1, updates_per_epoch * EPOCHS)
warmup = max(1, int(0.10 * num_update_steps))
lr_end = max(LR * 0.10, 1e-8)
scheduler = get_polynomial_decay_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup, num_training_steps=num_update_steps, lr_end=lr_end, power=1.0
)

use_amp = (device == "cuda")
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

# ----------------------- Baseline -----------------------
print("\n[VAL] Evaluating pretrained CE on VAL …")
baseline_list = evaluate_ndcg_groups(model, pretok_val, k=VAL_EVAL_K, eval_bs_pairs=EVAL_BATCH_PAIRS)
baseline = _mean_or_zero(baseline_list)
print(f"[VAL] Baseline NDCG@{VAL_EVAL_K}: {baseline:.4f}")
_write_metrics_line(METRICS_LOG, {"epoch": 0, "val_ndcg": round(baseline, 6), "is_best": False, "path": None})

best = baseline
best_ep = 0

# ----------------------- Train loop -----------------------
for ep in range(1, EPOCHS+1):
    model.train()
    running, micro = 0.0, 0
    t0 = time.perf_counter()

    for step, (enc_cpu, gains_cpu, spans, _pids) in enumerate(train_loader, 1):
        enc = {k: v.to(device, non_blocking=True) for k, v in enc_cpu.items()}
        gains = gains_cpu.to(device, non_blocking=True)

        if use_amp:
            ctx = torch.autocast(device_type="cuda", dtype=torch.float16)
        else:
            class _NoOp:
                def __enter__(self): pass
                def __exit__(self, *args): return False
            ctx = _NoOp()

        with ctx:
            logits = model(**enc).logits
            if logits.dim()==2 and logits.shape[1]==1: s = logits.squeeze(-1).float()
            elif logits.dim()==2 and logits.shape[1]==2: s = logits[:,1].float()
            else: s = logits.view(-1).float()
            loss = listnet_loss(s, gains, spans, tau=TAU)

        if scaler is not None and use_amp:
            scaler.scale(loss).backward()
            if CLIP_NORM > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
            if step % GRAD_ACCUM == 0:
                scaler.step(optimizer); scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
        else:
            loss.backward()
            if CLIP_NORM > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
            if step % GRAD_ACCUM == 0:
                optimizer.step(); scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        running += float(loss.item()); micro += 1
        if step % 50 == 0 or step == len(train_loader):
            elapsed = time.perf_counter() - t0
            print(f"[train e{ep}/{EPOCHS}] step {step}/{len(train_loader)} "
                  f"loss(avg)={running/micro:.4f}  lr={optimizer.param_groups[0]['lr']:.2e} "
                  f"steps/s={step/max(1e-6,elapsed):.2f}")

    # ----- Validate -----
    val_list = evaluate_ndcg_groups(model, pretok_val, k=VAL_EVAL_K, eval_bs_pairs=EVAL_BATCH_PAIRS)
    val_ndcg = _mean_or_zero(val_list)
    print(f"[VAL] epoch {ep}  NDCG@{VAL_EVAL_K}={val_ndcg:.4f}  (baseline {baseline:.4f})")

    # Save epoch checkpoint
    tag = f"epoch{ep:02d}_ndcg{val_ndcg:.4f}"
    ckpt_path = os.path.join(CHECKPOINTS, tag)
    _save_fp16_and_report(model, tok, ckpt_path)
    _write_metrics_line(METRICS_LOG, {"epoch": ep, "val_ndcg": round(val_ndcg, 6), "is_best": False, "path": ckpt_path})

    # Update best
    if val_ndcg > best + 1e-4:
        best = val_ndcg; best_ep = ep
        tmp_best = BEST_DIR + ".tmp"
        _replace_dir(tmp_best)
        _save_fp16_and_report(model, tok, tmp_best)
        if os.path.isdir(BEST_DIR): shutil.rmtree(BEST_DIR)
        os.rename(tmp_best, BEST_DIR)
        _write_metrics_line(METRICS_LOG, {"epoch": ep, "val_ndcg": round(val_ndcg, 6), "is_best": True, "path": BEST_DIR})

print(f"\n[RESULT] Best NDCG@{VAL_EVAL_K}: {best:.4f} (epoch {best_ep})")
print(f"[RESULT] best checkpoint → {BEST_DIR if os.path.isdir(BEST_DIR) else 'N/A'}")
print(f"[RESULT] per-epoch checkpoints → {CHECKPOINTS}")

# Cleanup
del model, tok, optimizer, scheduler, scaler, train_ds, train_loader, pretok_val
torch.cuda.empty_cache(); gc.collect()
